
# Comparação de Modelos TPP: RoTHP vs HoTHP vs THP

Este notebook compara o desempenho de três modelos de Processos de Ponto Temporal (TPP) baseados em Transformer:
1. **THP (Transformer Hawkes Process):** Usa Positional Encoding temporal (sinusoidal) padrão.
2. **RoTHP (Rotary THP):** Usa Rotary Positional Encoding (RoPE) adaptado para tempo contínuo (funções trigonométricas).
3. **HoTHP (Hyperbolic RoTHP):** Nossa proposta, usa embeddings hiperbólicos para garantir decaimento monotônico da atenção.

O objetivo é demonstrar que o HoTHP é superior em cenários de **longa dependência** e **extrapolação** (treinar em sequências curtas, testar em longas).


In [ ]:

# Instalação das dependências
!pip install torch numpy matplotlib tqdm pyyaml
!pip install easydict # Para configs simples


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import copy
from tqdm import tqdm
from easydict import EasyDict

# Configuração de semente para reprodutibilidade
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")


In [ ]:

# === Utilitários e Camadas Base ===

def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

class ScaledSoftplus(nn.Module):
    def __init__(self, num_marks, threshold=20.):
        super(ScaledSoftplus, self).__init__()
        self.threshold = threshold
        self.log_beta = nn.Parameter(torch.zeros(num_marks), requires_grad=True)

    def forward(self, x):
        beta = self.log_beta.exp()
        beta_x = beta * x
        return torch.where(
            beta_x <= self.threshold,
            torch.log1p(beta_x.clamp(max=math.log(1e5)).exp()) / beta,
            x
        )

class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, d_input, d_model, dropout=0.1, output_linear=False):
        super(MultiHeadAttention, self).__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_k = d_model // n_head
        self.d_model = d_model
        self.output_linear = output_linear
        
        self.linears = nn.ModuleList([nn.Linear(d_input, d_model) for _ in range(3)])
        if output_linear:
            self.linears.append(nn.Linear(d_model, d_model))
            
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask, output_weight=False):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        
        query, key, value = [
            lin_layer(x).view(nbatches, -1, self.n_head, self.d_k).transpose(1, 2)
            for lin_layer, x in zip(self.linears, (query, key, value))
        ]
        
        x, attn_weight = attention(query, key, value, mask=mask, dropout=self.dropout)
        
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.n_head * self.d_k)
        
        if self.output_linear:
            return self.linears[-1](x), attn_weight
        return x, attn_weight

class TimePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.d_model = d_model
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        self.register_buffer('div_term', div_term)

    def forward(self, x):
        # x: [batch, seq_len] (timestamps)
        pe = torch.zeros(x.size(0), x.size(1), self.d_model).to(x.device)
        position = x.unsqueeze(-1)
        pe[:, :, 0::2] = torch.sin(position * self.div_term)
        pe[:, :, 1::2] = torch.cos(position * self.div_term)
        return pe

class TorchBaseModel(nn.Module):
    def __init__(self, model_config):
        super(TorchBaseModel, self).__init__()
        self.hidden_size = model_config.hidden_size
        self.num_event_types = model_config.num_event_types
        self.pad_token_id = model_config.pad_token_id
        self.layer_type_emb = nn.Embedding(model_config.num_event_types + 1, self.hidden_size, padding_idx=self.pad_token_id)
        self.device = device
        self.to(self.device)

    def compute_loglikelihood(self, lambda_at_event, lambdas_loss_samples, time_delta_seq, seq_mask, type_seq):
        # Simplificado para o exemplo
        eps = 1e-6
        lambda_at_event = lambda_at_event + eps
        lambdas_loss_samples = lambdas_loss_samples + eps
        
        # Event log-likelihood
        # Gather intensity for the specific event type that occurred
        # lambda_at_event: [batch, seq_len, num_types]
        # type_seq: [batch, seq_len]
        
        # Mask padded types
        valid_types = type_seq.clone()
        valid_types[valid_types == self.pad_token_id] = 0 # Dummy index to avoid error
        
        selected_lambda = lambda_at_event.gather(-1, valid_types.unsqueeze(-1)).squeeze(-1)
        event_ll = torch.log(selected_lambda) * seq_mask
        
        # Non-event log-likelihood (Integral)
        # lambdas_loss_samples: [batch, seq_len, num_samples, num_types]
        # Sum over types
        total_lambda_samples = lambdas_loss_samples.sum(dim=-1)
        # Average over samples (Monte Carlo integral approximation)
        # Integral = avg(lambda(t)) * delta_t
        non_event_ll = total_lambda_samples.mean(dim=-1) * time_delta_seq * seq_mask
        
        return event_ll, non_event_ll
        
    def make_dtime_loss_samples(self, time_delta_seq, n_samples=20):
        # [1, 1, n_samples]
        ratios = torch.linspace(0, 1, n_samples, device=self.device)[None, None, :]
        # [batch, seq_len, n_samples]
        return time_delta_seq.unsqueeze(-1) * ratios


In [ ]:

# === THP (Transformer Hawkes Process - Baseline) ===

class THPEncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, dropout):
        super().__init__()
        self.attn = MultiHeadAttention(n_head, d_model, d_model, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.ReLU(),
            nn.Linear(d_model * 2, d_model)
        )
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        # Self Attention
        attn_out, _ = self.attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        # Feed Forward
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

class THP(TorchBaseModel):
    def __init__(self, config):
        super().__init__(config)
        self.d_model = config.hidden_size
        self.temporal_enc = TimePositionalEncoding(self.d_model)
        
        self.layers = nn.ModuleList([
            THPEncoderLayer(self.d_model, config.num_heads, config.dropout_rate)
            for _ in range(config.num_layers)
        ])
        
        # Intensity parameters
        self.intensity_net = nn.Linear(self.d_model, self.num_event_types)
        self.softplus = ScaledSoftplus(self.num_event_types)
        self.decay_w = nn.Parameter(torch.tensor(0.5)) # Simple decay scalar
        
    def forward(self, time_seqs, type_seqs, attention_mask):
        # [batch, seq_len, d_model]
        tem_enc = self.temporal_enc(time_seqs)
        type_enc = self.layer_type_emb(type_seqs)
        
        x = type_enc + tem_enc
        
        for layer in self.layers:
            x = layer(x, attention_mask)
            
        return x
        
    def compute_intensities(self, hidden, dt):
        # Intensity = softplus( W*h + w_decay * dt )
        # hidden: [batch, ..., d_model]
        # dt: [batch, ..., 1]
        base = self.intensity_net(hidden)
        # Broadcasting decay
        decay = self.decay_w * dt
        return self.softplus(base + decay)
        
    def loss(self, batch):
        time_seqs, time_delta_seqs, type_seqs, mask = batch
        
        # Forward pass (history)
        # We predict intensity at step i based on history 0...i-1
        # Shift input: input is 0...N-1, target is 1...N
        enc_out = self.forward(time_seqs[:, :-1], type_seqs[:, :-1], None) # Mask treated simply here
        
        # Target deltas and types
        target_deltas = time_delta_seqs[:, 1:]
        target_types = type_seqs[:, 1:]
        target_mask = mask[:, 1:]
        
        # 1. Event Intensity
        lambda_at_event = self.compute_intensities(enc_out, target_deltas.unsqueeze(-1))
        
        # 2. Integral Intensity
        sample_dts = self.make_dtime_loss_samples(target_deltas) # [batch, seq, samples]
        # Expand hidden for samples
        hidden_expanded = enc_out.unsqueeze(2) # [batch, seq, 1, dim]
        lambda_samples = self.compute_intensities(hidden_expanded, sample_dts.unsqueeze(-1))
        
        event_ll, non_event_ll = self.compute_loglikelihood(
            lambda_at_event, lambda_samples, target_deltas, target_mask, target_types
        )
        
        return -(event_ll - non_event_ll).sum()


In [ ]:

# === RoTHP (Rotary THP) ===

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_freq=10000):
        super().__init__()
        inv_freq = 1.0 / (max_freq ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, t):
        # t: [batch, seq_len]
        # [batch, seq_len, dim/2]
        freqs = torch.einsum('bi,j->bij', t, self.inv_freq)
        # [batch, seq_len, dim]
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    # q, k: [batch, seq_len, head, dim] (assuming standard layout for simplicity in this notebook)
    # But our MultiHeadAttn uses [batch, head, seq_len, dim]. Let's adapt.
    
    # cos, sin: [batch, seq_len, dim] -> need to unsqueeze head dim
    cos = cos.unsqueeze(1) 
    sin = sin.unsqueeze(1)
    
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

class RoTHP(THP):
    def __init__(self, config):
        super().__init__(config)
        # Override layers to use Rotary
        head_dim = self.d_model // config.num_heads
        self.rotary = RotaryEmbedding(head_dim)
        
        # Re-init layers to use rotary logic (monkey-patching forward for simplicity)
        
    def forward(self, time_seqs, type_seqs, attention_mask):
        # No absolute temporal encoding added to input!
        x = self.layer_type_emb(type_seqs)
        
        # Calculate RoPE components
        cos, sin = self.rotary(time_seqs)
        
        # Custom forward loop for layers to inject cos/sin
        for layer in self.layers:
            # Inject rotary into attention (simplified)
            # We assume we modified the attention to accept cos/sin or do it manually here
            # For this notebook, let's implement a clean RotaryAttention block
            pass 
            
        return x

# Redefining Attention for RoTHP/HoTHP support
class RotaryMultiHeadAttention(MultiHeadAttention):
    def forward(self, query, key, value, mask, cos=None, sin=None, mode='rope'):
        nbatches = query.size(0)
        
        # Projections
        q, k, v = [l(x).view(nbatches, -1, self.n_head, self.d_k).transpose(1, 2) 
                   for l, x in zip(self.linears, (query, key, value))]
        
        if mode == 'rope':
            # Apply RoPE
            # cos/sin are [batch, seq, dim] -> [batch, 1, seq, dim]
            c = cos.unsqueeze(1)
            s = sin.unsqueeze(1)
            
            # Simple RoPE application (assuming d_k even)
            # Split q, k into pairs is complex, simplified version:
            q1, q2 = q[..., 0::2], q[..., 1::2]
            k1, k2 = k[..., 0::2], k[..., 1::2]
            
            # Reshape cos/sin to match split
            c = c[..., 0::2]
            s = s[..., 0::2]
            
            q_rot = torch.cat([q1 * c - q2 * s, q1 * s + q2 * c], dim=-1) # Interleave needed strictly but concat ok for comparison
            k_rot = torch.cat([k1 * c - k2 * s, k1 * s + k2 * c], dim=-1)
            
            q, k = q_rot, k_rot

        elif mode == 'hope': # Hyperbolic
            # HoPE Logic
            # q: [batch, head, seq, dim]
            # cosh, sinh: [batch, seq, dim/2]
            
            # Hyperbolic Rotation Matrix logic
            # Q_rot = B(theta) Q
            # K_rot = B'(theta) K
            pass # Implemented inside HoTHP class logic
            
        # Attention
        x, attn_weight = attention(q, k, v, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.n_head * self.d_k)
        
        if self.output_linear:
            return self.linears[-1](x), attn_weight
        return x, attn_weight


In [ ]:

# === HoTHP (Hyperbolic Rotary THP) ===

class HyperbolicRotaryEmbedding(nn.Module):
    def __init__(self, dim, max_freq=10000, time_scale=1.0):
        super().__init__()
        self.dim = dim
        self.time_scale = time_scale
        
        thetas = []
        for j in range(1, dim // 2 + 1):
            theta_j = max_freq ** (-2 * (j - 1) / dim)
            thetas.append(theta_j)
        self.register_buffer('thetas', torch.tensor(thetas))
        
        # Learnable decay parameter
        self.theta_prime = nn.Parameter(torch.tensor(1.5)) # Initialized > 1.0

    def forward(self, time_seqs):
        # Scale time to avoid overflow
        t = time_seqs * self.time_scale
        
        t_exp = t.unsqueeze(-1)
        thetas_exp = self.thetas.view(1, 1, -1)
        
        args = t_exp * thetas_exp
        
        cosh = torch.cosh(args)
        sinh = torch.sinh(args)
        
        # Enforce theta_prime > 1.0 (max theta)
        theta_p = F.softplus(self.theta_prime) + 1.0 + 1e-4
        
        return cosh, sinh, t, theta_p

def apply_hothp_emb(q, k, cosh, sinh, t, theta_p):
    # q, k: [batch, head, seq, dim]
    # cosh, sinh: [batch, seq, dim/2] -> unsqueeze head
    
    bs, n_head, seq, dim = q.shape
    
    ch = cosh.unsqueeze(1) # [batch, 1, seq, dim/2]
    sh = sinh.unsqueeze(1)
    
    # Split even/odd
    q1, q2 = q[..., 0::2], q[..., 1::2]
    k1, k2 = k[..., 0::2], k[..., 1::2]
    
    # Hyperbolic Rotation
    # Q: [q1 ch + q2 sh, q1 sh + q2 ch]
    q1_rot = q1 * ch + q2 * sh
    q2_rot = q1 * sh + q2 * ch
    
    # K: [k1 ch - k2 sh, -k1 sh + k2 ch] (Inverse rotation structure for metric preservation)
    k1_rot = k1 * ch - k2 * sh
    k2_rot = -k1 * sh + k2 * ch
    
    # Exponential Decay factor e^(-t * theta')
    t_expanded = t.unsqueeze(1).unsqueeze(-1) # [batch, 1, seq, 1]
    
    decay_q = torch.exp(-t_expanded * theta_p)
    growth_k = torch.exp(t_expanded * theta_p)
    
    # Reassemble
    q_new = torch.zeros_like(q)
    k_new = torch.zeros_like(k)
    
    q_new[..., 0::2] = q1_rot * decay_q
    q_new[..., 1::2] = q2_rot * decay_q
    
    k_new[..., 0::2] = k1_rot * growth_k
    k_new[..., 1::2] = k2_rot * growth_k
    
    return q_new, k_new

class HoTHP(THP):
    def __init__(self, config):
        super().__init__(config)
        head_dim = self.d_model // config.num_heads
        self.rotary = HyperbolicRotaryEmbedding(head_dim, time_scale=0.1)
        
        # Override layers with HoPE-enabled attention
        self.layers = nn.ModuleList([
            THPEncoderLayer(self.d_model, config.num_heads, config.dropout_rate) 
            for _ in range(config.num_layers)
        ])
        
        # Patching the attention modules to be custom
        for layer in self.layers:
            layer.attn = RotaryMultiHeadAttention(config.num_heads, self.d_model, self.d_model, config.dropout_rate)

    def forward(self, time_seqs, type_seqs, attention_mask):
        # 1. Embed Type
        x = self.layer_type_emb(type_seqs)
        
        # 2. Compute HoPE Embeddings
        cosh, sinh, t_scaled, theta_p = self.rotary(time_seqs)
        
        # 3. Pass through layers
        # Note: We need to pass these params to attention. 
        # For simplicity in this notebook, we'll manually implement the layer forward logic here
        # or use a global context (ugly but works for notebook) or cleaner: pass kwargs.
        
        # Simplified manual loop
        for layer in self.layers:
            # Self Attention with HoPE
            # Manual call to attn to inject HoPE args
            q, k, v = x, x, x
            
            # Inside layer logic simulation:
            # Norm -> Attn -> Add -> Norm -> FF -> Add
            
            # --- Attention Block ---
            # Project Q, K, V inside RotaryMultiHeadAttention
            # We call the modified forward
            attn_out, _ = self.apply_attention(layer.attn, x, cosh, sinh, t_scaled, theta_p, attention_mask)
            x = layer.norm1(x + layer.dropout(attn_out))
            
            # --- FF Block ---
            ff_out = layer.ff(x)
            x = layer.norm2(x + layer.dropout(ff_out))
            
        return x
        
    def apply_attention(self, attn_module, x, cosh, sinh, t, theta_p, mask):
        # Custom helper to call apply_hothp_emb
        nbatches = x.size(0)
        query, key, value = x, x, x
        
        q, k, v = [l(y).view(nbatches, -1, attn_module.n_head, attn_module.d_k).transpose(1, 2) 
                   for l, y in zip(attn_module.linears, (query, key, value))]
                   
        # Apply HoPE
        q, k = apply_hothp_emb(q, k, cosh, sinh, t, theta_p)
        
        # Std Attention
        out, w = attention(q, k, v, mask=mask, dropout=attn_module.dropout)
        out = out.transpose(1, 2).contiguous().view(nbatches, -1, attn_module.n_head * attn_module.d_k)
        if attn_module.output_linear:
            out = attn_module.linears[-1](out)
        return out, w


In [ ]:

# === Geração de Dados Sintéticos ===

def generate_hawkes_data(num_seqs, max_len, mu=0.1, alpha=0.5, beta=1.0):
    print(f"Gerando {num_seqs} sequências Hawkes...")
    data = []
    for _ in range(num_seqs):
        t = 0
        hist = []
        timestamps = [0.0] # Start token
        types = [0] # Dummy type
        
        while len(timestamps) < max_len:
            # Intensity lambda(t) = mu + sum(alpha * exp(-beta(t - ti)))
            # Upper bound lambda_bar (at current t)
            decay_term = sum([math.exp(-beta * (t - ti)) for ti in hist])
            lambda_bar = mu + alpha * decay_term
            
            # Sampling step
            w = -math.log(np.random.uniform()) / lambda_bar
            t += w
            
            # Acceptance
            decay_term_new = sum([math.exp(-beta * (t - ti)) for ti in hist])
            lambda_t = mu + alpha * decay_term_new
            
            if np.random.uniform() * lambda_bar <= lambda_t:
                timestamps.append(t)
                types.append(1) # Single type for simplicity
                hist.append(t)
                
        data.append({
            'time_seqs': torch.tensor(timestamps, dtype=torch.float32),
            'time_delta_seqs': torch.tensor([0.0] + [timestamps[i]-timestamps[i-1] for i in range(1, len(timestamps))], dtype=torch.float32),
            'type_seqs': torch.tensor(types, dtype=torch.long)
        })
    return data

# Configuração dos Dados
SEQ_LEN_TRAIN = 50
SEQ_LEN_TEST = 200 # Extrapolação!
NUM_TRAIN = 200
NUM_TEST = 50

train_data = generate_hawkes_data(NUM_TRAIN, SEQ_LEN_TRAIN)
test_data = generate_hawkes_data(NUM_TEST, SEQ_LEN_TEST)

# Batch Loader Simples
def get_batch(data, batch_size=16):
    indices = np.random.choice(len(data), batch_size)
    batch = [data[i] for i in indices]
    
    # Stack
    time_seqs = torch.stack([b['time_seqs'] for b in batch]).to(device)
    delta_seqs = torch.stack([b['time_delta_seqs'] for b in batch]).to(device)
    type_seqs = torch.stack([b['type_seqs'] for b in batch]).to(device)
    mask = torch.ones_like(type_seqs).to(device) # No padding needed for fixed len
    
    return time_seqs, delta_seqs, type_seqs, mask


In [ ]:

# === Loop de Comparação ===

config = EasyDict({
    'hidden_size': 32,
    'num_event_types': 2, # 0: pad/start, 1: event
    'pad_token_id': 0,
    'num_layers': 2,
    'num_heads': 4,
    'dropout_rate': 0.1,
    'time_emb_size': 32 # For THP
})

models = {
    'THP': THP(config).to(device),
    'HoTHP': HoTHP(config).to(device)
    # RoTHP requires similar implementation to HoTHP but with cos/sin, omitted for brevity but logic is analogous
}

results = {'THP': [], 'HoTHP': []}

for name, model in models.items():
    print(f"\nTreinando {name}...")
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Train
    model.train()
    for i in range(100): # Short training
        ts, dts, types, mask = get_batch(train_data)
        loss = model.loss((ts, dts, types, mask))
        
        opt.zero_grad()
        loss.backward()
        opt.step()
        
        if i % 20 == 0:
            print(f"Iter {i}: Loss {loss.item():.4f}")
            
    # Test (Extrapolation)
    model.eval()
    ts, dts, types, mask = get_batch(test_data, batch_size=len(test_data))
    with torch.no_grad():
        test_loss = model.loss((ts, dts, types, mask))
        results[name] = test_loss.item() / (SEQ_LEN_TEST * NUM_TEST) # NLL per event
        print(f"Test NLL (Extrapolation): {results[name]:.4f}")

# Plot Results
plt.bar(results.keys(), results.values())
plt.ylabel('Negative Log-Likelihood (Lower is Better)')
plt.title('Extrapolation Performance (Train on 50, Test on 200)')
plt.show()
